# Etapa 4.1 Redis

In [29]:
import redis
# conectamos a la instancia local
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 4.1.1 
## Datos de usuarios

Los datos de usuario son datos que en general no cambian y que consultamos con mucha recurrencia, por ejemplo en la sesión de un usuario en una aplicación siempre aparecen sus datos.

In [30]:
# importamos pandas para leer el archivo CSV
import pandas as pd
# datos de usuarios
usuarios = pd.read_csv('usuarios.csv')

print(usuarios.info()) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID              1000 non-null   int64 
 1   Nombre          1000 non-null   object
 2   Email           1000 non-null   object
 3   Telefono        1000 non-null   object
 4   Fecha_Registro  1000 non-null   object
dtypes: int64(1), object(4)
memory usage: 39.2+ KB
None


In [31]:
# Cargo los datos
# Conservo los indices de la tabla que sé que están en orden del 1 al 1000
# Guardo los datos de cada usuario en un hash
for i in range(0,1000):
    r.hset(f"usuario:{usuarios['ID'].iloc[i]}", mapping={"Nombre": usuarios['Nombre'].iloc[i],
                                                         "Email": usuarios['Email'].iloc[i],
                                                         "Telefono": usuarios['Telefono'].iloc[i],
                                                         "Fecha_Registro": usuarios['Fecha_Registro'].iloc[i]})

In [32]:
# consulto los datos del usuario 333
print("Datos del usuario 333:")
print(r.hgetall("usuario:333"),'\n')

# vamos a revisar solamente su teléfono
print("Teléfono del usuario 333:")
print(r.hget("usuario:333", "Telefono"),'\n')

# este número está desactualizado, vamos a modificarlo
r.hset("usuario:333", "Telefono", "+54 15 2606 2222")

# consulto nuevamente el teléfono de la usuaria 333
print("Teléfono actualizado del usuario 333:")
print(r.hget("usuario:333", "Telefono"))

Datos del usuario 333:
{'Nombre': 'Olivia Villalba', 'Email': 'olivia.villalba333@gmail.com', 'Telefono': '+54 15 2629 7379', 'Fecha_Registro': '2025-10-11'} 

Teléfono del usuario 333:
+54 15 2629 7379 

Teléfono actualizado del usuario 333:
+54 15 2606 2222


## Estado de los pedidos 

El estado de un pedido cambia constantemente a lo largo de su ciclo de vida (pendiente, en preparación, en camino, entregado). Al ser un único valor simple que se consulta y actualiza con mucha frecuencia, lo guardamos como un par clave-valor.

Cargamos el historial de estados de los pedidos desde el CSV para luego volcar en Redis el estado más reciente de cada uno.

In [33]:
historial_estado_pedidos = pd.read_csv('historial_estado_pedido.csv')

# mostramos la información del DataFrame
print(historial_estado_pedidos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18894 entries, 0 to 18893
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_historial  18894 non-null  int64 
 1   id_pedido     18894 non-null  int64 
 2   estado        18894 non-null  object
 3   fecha_hora    18894 non-null  object
dtypes: int64(2), object(2)
memory usage: 590.6+ KB
None


In [34]:
# como sé que el historial esta en orden por pedido y por hora, actualizo el estado de cada pedido iterando
# sobre la tabla de historial, cada pedido se actualizará hasta el último que alcanzó
for i in range(0,18894):
    r.set(f"idpedido:{historial_estado_pedidos['id_pedido'].iloc[i]}", historial_estado_pedidos['estado'].iloc[i])

Consultamos el estado de algunos pedidos y simulamos una transición: el pedido 187 pasa a "en camino" cuando el repartidor lo retira del restaurante.

In [35]:
# veamos el estado del pedido 1000
print("Estado del pedido 1000:")
print(r.get("idpedido:1000"),'\n')

# veamos el estado del pedido 1001
print("Estado del pedido 187:")
print(r.get("idpedido:187"),'\n')

# el pedido 187 ya fue entregado al delivery y está en camino, vamos a cambiar el estado a en_camino
r.set("idpedido:187", "en_camino")

# veamos nuevamente el estado del pedido 187
print("el pedido 187 fue preparado y ya está en camino\n")
print("Estado del pedido 187:")
print(r.get("idpedido:187"))

Estado del pedido 1000:
entregado 

Estado del pedido 187:
en_preparacion 

el pedido 187 fue preparado y ya está en camino

Estado del pedido 187:
en_camino


## Estado de los repartidores

El estado del repartidor (activo / inactivo) es un dato volátil y de alta frecuencia de acceso: el sistema lo consulta cada vez que necesita asignar un pedido. Lo modelamos como clave-valor por ser un único string simple.

Cargamos la tabla de repartidores desde el CSV para registrar su estado operacional en Redis.

In [36]:
repartidores = pd.read_csv('repartidores.csv')

repartidores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_repartidor  200 non-null    int64 
 1   Nombre         200 non-null    object
 2   Telefono       200 non-null    object
 3   Transporte     200 non-null    object
 4   Estado         200 non-null    object
dtypes: int64(1), object(4)
memory usage: 7.9+ KB


In [37]:
# cargo como tuplas clave:valor el estado del repartidor, si está activo es que no está haciendo ningún pedido,
# si está inactivo es que está haciendo un pedido
for i in range(0,200):
    r.set(f"id_repartidor:{repartidores['id_repartidor'].iloc[i]}", repartidores['Estado'].iloc[i])

Simulamos un cambio de estado: el repartidor 111 toma un pedido y pasa de activo a inactivo.

In [38]:
# consultemos el estado del repartidor con el id 111
print("Estado del repartidor 111:")
print(r.get("id_repartidor:111"),'\n')

# como el repartidor 111 tomó el pedido 178 ahora este ya no está activo
# cambiemos el estado a inactivo
r.set("id_repartidor:111", "inactivo")

print("El repartidor 111 tomó el pedido 178\n")
print ("Estado del repartidor 111:")
print(r.get("id_repartidor:111"),'\n')

Estado del repartidor 111:
activo 

El repartidor 111 tomó el pedido 178

Estado del repartidor 111:
inactivo 



# 4.1.2 — Listas

### Historial de restaurantes vistos recientemente

Nuestro sistema quiere proporcionar al usuario un historial de restaurantes vistos (en una aplicación o págiona web) recientemente.

Aquí mostramos una simulación de cómo se implementaría para un usuario en particular. Como solo nos interesa guardar los
últimos restaurantes visitados, el sistema guarda hasta 10 y va eliminando los más antiguos.

In [39]:
# simulamos la sesión de un usuario
id_usuario = 438
clave_historial = f"historial_vistos:usuario:{id_usuario}"

r.delete(clave_historial)  # limpiamos la clave por si se ejecuta esta celda varias veces

# En nuestra simulacion el usuario visita los siguientes restaurantes
restaurantes_visitados = [15, 3, 27, 8, 15, 42, 9, 33, 11, 5, 42, 18, 22]

print(f"--- Simulando navegación del usuario {id_usuario} ---")
for rest in restaurantes_visitados:
    # comienzo a registrar las visitas
    r.lpush(clave_historial, rest)
    
    # Si la longitud de la lista supera los 10 elementos, elimina el más antiguo
    if len(r.lrange(clave_historial, 0, -1)) > 10:
        r.rpop(clave_historial)
    
    print(f"El usuario entró al restaurante {rest}")

# consultamos la longitud para verificar que el límite de 10 funcionó perfectamente
longitud = r.llen(clave_historial)
print(f"\nLongitud actual del historial de navegación: {longitud} restaurantes.",'\n')

# mostremos el historial de los restaurantes visitados recientemente
print(f"Cinta de 'vistos recientemente' enviada a la interfaz: {r.lrange(clave_historial, 0, -1)}")

# ahora el usuario quiere eliminar por alguna razón el último restaurante que visitó, y también el restaurante 33
print(f"\nSe eliminó el último restaurante que visitó: {r.lpop(clave_historial)}")
r.lrem(clave_historial,1,33)
print("Se eliminó el restaurante 33 del historial \n")

cinta_actualizada = r.lrange(clave_historial, 0, -1)
print(f"Cinta actualizada tras la eliminación: {cinta_actualizada}")

--- Simulando navegación del usuario 438 ---
El usuario entró al restaurante 15
El usuario entró al restaurante 3
El usuario entró al restaurante 27
El usuario entró al restaurante 8
El usuario entró al restaurante 15
El usuario entró al restaurante 42
El usuario entró al restaurante 9
El usuario entró al restaurante 33
El usuario entró al restaurante 11
El usuario entró al restaurante 5
El usuario entró al restaurante 42
El usuario entró al restaurante 18
El usuario entró al restaurante 22

Longitud actual del historial de navegación: 10 restaurantes. 

Cinta de 'vistos recientemente' enviada a la interfaz: ['22', '18', '42', '5', '11', '33', '9', '42', '15', '8']

Se eliminó el último restaurante que visitó: 22
Se eliminó el restaurante 33 del historial 

Cinta actualizada tras la eliminación: ['18', '42', '5', '11', '9', '42', '15', '8']


# 4.1.3 — TTL: Datos con tiempo de vida

In [40]:
import time

### Tiempo de vida de promociones

Los restaurantes publican promociones que deben dejar de estar vigentes automáticamente al cabo de un tiempo. Implementamos una función que verifica el tiempo restante de una promoción usando el TTL de su clave.

In [41]:
# simulamos una promoción publicada por un restaurante
id_restaurante = 15
clave_promocion = f"promocion:restaurante:{id_restaurante}"

# Publicamos una promoción del 20% con vigencia por un día (86400 segundos)
r.set(clave_promocion, "20%", ex=86400)
promocion = r.get(clave_promocion)
print(f"Restaurante {id_restaurante} publicó una promoción de {promocion}\n")

Restaurante 15 publicó una promoción de 20%



In [42]:
def verificar_tiempo_promocion(id_restaurante):
    clave_promocion = f"promocion:restaurante:{id_restaurante}"
    promocion = r.get(clave_promocion)
    tiempo_prom = r.ttl(clave_promocion)

    if promocion is None:
        print(f"La promoción del restaurante {id_restaurante} ha expirado.")
    else:
        print(f"La promoción del restaurante {id_restaurante} expira en {tiempo_prom} segundos.")

In [43]:
# Creamos una promoción ficticia con un tiempo de expiración de 2 segundos
id_restaurante = 5
r.set(f"promocion:restaurante:{id_restaurante}", "20%", ex=2)
print("Promoción ficticia creada con un tiempo de expiración de 2 segundos.")

# Verificamos mientras la promoción sigue vigente
verificar_tiempo_promocion(id_restaurante)

# Esperamos a que expire
time.sleep(3)

# Verificamos nuevamente: ahora ya expiró
verificar_tiempo_promocion(id_restaurante)

Promoción ficticia creada con un tiempo de expiración de 2 segundos.
La promoción del restaurante 5 expira en 2 segundos.


La promoción del restaurante 5 ha expirado.


### Tiempo para aceptar un pedido

Cuando el sistema le ofrece un pedido a un repartidor, este tiene una ventana de tiempo para aceptarlo. Si no responde antes de que la oferta expire, la clave se elimina sola y el pedido puede ofrecerse a otro repartidor.

In [44]:
id_repartidor = 111
id_pedido = 9901
clave_oferta = f"oferta_pedido:repartidor:{id_repartidor}"

# El sistema le ofrece el pedido 9901 al repartidor 111, tiene 60 segundos para aceptar
r.set(clave_oferta, id_pedido, ex=60)
print(f"Se le ofreció el pedido {id_pedido} al repartidor {id_repartidor}. Tiene 60 segundos para aceptar.")

Se le ofreció el pedido 9901 al repartidor 111. Tiene 60 segundos para aceptar.


Si el repartidor 111 acepta antes de que la oferta expire, se consume la clave y su estado pasa a inactivo. Si no, la oferta ya habrá expirado y el pedido se ofrecerá a otro repartidor.

In [45]:
if r.ttl(clave_oferta) > 0:  # verificamos si la oferta sigue vigente
    print(f"El repartidor {id_repartidor} aceptó el pedido {id_pedido}.")
    r.delete(clave_oferta)              # consumimos la oferta
    r.set(f"id_repartidor:{id_repartidor}", "inactivo")  # ahora está ocupado
else:
    print("La oferta ya había expirado.")

El repartidor 111 aceptó el pedido 9901.


### Tiempo límite de preparación

Una vez realizado el pedido, el restaurante tiene 25 minutos para entregarlo al repartidor. Si ese tiempo se agota y el pedido sigue en preparación, el sistema actualiza su estado a "en_preparacion(demorado)".

Es importante notar que Redis no ejecuta el cambio de estado por sí solo: únicamente provee el temporizador. Quien detecta la demora es el sistema cuando consulta el pedido y observa que la clave de tiempo ya expiró.

In [46]:
id_pedido = 9901
clave_prep = f"tiempo_preparacion:pedido:{id_pedido}"

# Al crear el pedido, registramos su estado y le damos 25 minutos de margen (1500 segundos)
r.set(f"idpedido:{id_pedido}", "en_preparacion")
r.set(clave_prep, "en_tiempo", ex=1500)

True

In [47]:
tiempo_restante = r.ttl(clave_prep)

print(f"Pedido {id_pedido} en preparación. El restaurante tiene {tiempo_restante // 60} minutos para entregarlo al repartidor.")
print(f"Tiempo restante: {tiempo_restante} segundos.")

Pedido 9901 en preparación. El restaurante tiene 25 minutos para entregarlo al repartidor.
Tiempo restante: 1500 segundos.


In [48]:
def verificar_tiempo(id_pedido):
    clave_prep = f"tiempo_preparacion:pedido:{id_pedido}"
    estado_actual = r.get(f"idpedido:{id_pedido}")

    # Si el pedido ya fue retirado o entregado, no aplica
    if estado_actual in ("en_camino", "entregado"):
        print(f"Pedido {id_pedido}: ya fue retirado por el repartidor. Sin demora.")
        return

    # La clave de tiempo expiró
    if r.ttl(clave_prep) == -2 and estado_actual == "en_preparacion":
        r.set(f"idpedido:{id_pedido}", "en_preparacion(demorado)")
        print(f"Pedido {id_pedido}: Estado en preparación demorado. Más de 25 minutos.")
    else:
        print(f"Pedido {id_pedido}: en tiempo. Quedan {r.ttl(clave_prep)} segundos.")


In [49]:
# Creamos un pedido con margen de 2 segundos para la demo
id_pedido = 9902
r.set(f"idpedido:{id_pedido}", "en_preparacion")
r.set(f"tiempo_preparacion:pedido:{id_pedido}", "en_tiempo", ex=2)

print("Recién creado:")
verificar_tiempo(id_pedido)

print("\nEsperamos a que pase el tiempo límite...")
time.sleep(3)

print("\nDespués de los 25 min (simulados):")
verificar_tiempo(id_pedido)

Recién creado:
Pedido 9902: en tiempo. Quedan 2 segundos.

Esperamos a que pase el tiempo límite...

Después de los 25 min (simulados):
Pedido 9902: Estado en preparación demorado. Más de 25 minutos.
